# Dusha Eval — 5×5 Cross-Evaluation Matrix

Загружает обученные чекпоинты из `aleksandribryanov/wavlm-dusha-checkpoints`,  
тестирует каждую из 5 моделей на каждом из 5 тестовых TSV.  
Для лучшей пары (модель, тест) выводит полные метрики и confusion matrix.

## 1. GPU check

In [ ]:
import subprocess, sys, os
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('CUDA not available')

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchaudio', 'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn', 'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Clone repo

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

## 4. Paths & checkpoints

In [ ]:
import warnings, logging, pathlib
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

# ── пути к датасетам ──────────────────────────────────────────────────────────
AGG_ROOT    = pathlib.Path('/kaggle/input/datasets/aleksandribryanov/agg-dusha')
AUDIO_BASE  = pathlib.Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd')
CKPT_DIR    = pathlib.Path('/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints')
TEST_AUDIO_DIR = str(AUDIO_BASE / 'crowd_test')
# ─────────────────────────────────────────────────────────────────────────────

AGGREGATIONS = {
    'majority': 'aggregated_majority_test.tsv',
    'ds_0.85':  'aggregated_ds_0.85_test.tsv',
    'ds_0.9':   'aggregated_ds_0.9_test.tsv',
    'ds_0.95':  'aggregated_ds_0.95_test.tsv',
    'ds_0.98':  'aggregated_ds_0.98_test.tsv',
}
TEST_TSVS = {tag: AGG_ROOT / tsv for tag, tsv in AGGREGATIONS.items()}

import pandas as pd
print(f'CKPT_DIR       : {CKPT_DIR}  exists={CKPT_DIR.exists()}')
print(f'TEST_AUDIO_DIR : {TEST_AUDIO_DIR}  exists={pathlib.Path(TEST_AUDIO_DIR).exists()}')
print('\nCheckpoints:')
for tag in AGGREGATIONS:
    p = CKPT_DIR / f'wavlm_dusha_{tag}.pt'
    print(f'  {tag:12s}  {"✓" if p.exists() else "✗"}  {p.name}')
print('\nTest TSVs:')
for tag, path in TEST_TSVS.items():
    n = len(pd.read_csv(path, sep='\t')) if path.exists() else 0
    print(f'  {tag:12s}  {"✓" if path.exists() else "✗"}  {n:6,} rows')

## 5. 5×5 Cross-Evaluation Matrix

In [ ]:
import gc
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor

from src.config import ExperimentConfig
from src.dataset import get_dusha_test_dataloader
from src.models import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def _config_from_ckpt(ckpt, audio_dir):
    cfg = ExperimentConfig()
    for k, v in ckpt.get('config', {}).items():
        if hasattr(cfg, k):
            setattr(cfg, k, v)
    cfg.audio_dir   = audio_dir
    cfg.num_workers = 2
    return cfg

tags = list(AGGREGATIONS.keys())
matrix     = np.full((len(tags), len(tags)), np.nan)  # [model_tag, test_tag]
all_results = {}  # (model_tag, test_tag) -> (preds, labels)

for i, model_tag in enumerate(tags):
    ckpt_path = CKPT_DIR / f'wavlm_dusha_{model_tag}.pt'
    if not ckpt_path.exists():
        print(f'[{model_tag}] checkpoint not found — skip')
        continue

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg  = _config_from_ckpt(ckpt, TEST_AUDIO_DIR)
    processor = AutoFeatureExtractor.from_pretrained(cfg.processor_name or cfg.model_name)

    model = build_model(cfg)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(device)
    print(f'\n[{model_tag}]  strategy={cfg.fine_tune_strategy}')

    for j, test_tag in enumerate(tags):
        test_tsv = TEST_TSVS[test_tag]
        if not test_tsv.exists():
            continue

        loader = get_dusha_test_dataloader(str(test_tsv), TEST_AUDIO_DIR, processor, cfg)
        preds_list, labels_list = [], []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f'  test={test_tag}', leave=False):
                inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                preds_list.append(model(**inputs).argmax(dim=-1).cpu().numpy())
                labels_list.append(batch['labels'].numpy())

        preds  = np.concatenate(preds_list)
        labels = np.concatenate(labels_list)
        wacc   = balanced_accuracy_score(labels, preds)
        matrix[i, j] = wacc
        all_results[(model_tag, test_tag)] = (preds, labels)
        print(f'  test={test_tag:12s}  wacc={wacc:.4f}')

    del model; gc.collect(); torch.cuda.empty_cache()

tags_short = [t.replace('ds_', '') for t in tags]
matrix_df = pd.DataFrame(matrix, index=tags, columns=tags)
matrix_df.index.name = 'model \ test'
matrix_df.to_csv('/kaggle/working/cross_eval_matrix.tsv', sep='\t', float_format='%.4f')
print('\nWeighted Accuracy Matrix:')
print(matrix_df.to_string(float_format='{:.4f}'.format))

In [ ]:
valid = matrix_df.values[~np.isnan(matrix_df.values)]
vmin = valid.min() - 0.005
vmax = valid.max() + 0.005

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrix_df.astype(float),
    annot=True, fmt='.3f', cmap='YlGn',
    vmin=vmin, vmax=vmax,
    xticklabels=tags, yticklabels=tags,
    ax=ax,
)
ax.set_xlabel('Tested on')
ax.set_ylabel('Trained on')
ax.set_title(f'Cross-evaluation: Weighted Accuracy  [{vmin:.3f} – {vmax:.3f}]')
plt.tight_layout()
plt.savefig('/kaggle/working/cross_eval_matrix.png', dpi=150)
plt.show()

## 6. Best pair — full metrics + confusion matrix

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)

# найти лучшую пару (model, test)
best_val, best_model_tag, best_test_tag = -1, None, None
for i, mt in enumerate(tags):
    for j, tt in enumerate(tags):
        if not np.isnan(matrix[i, j]) and matrix[i, j] > best_val:
            best_val, best_model_tag, best_test_tag = matrix[i, j], mt, tt

print(f'Best pair:  model={best_model_tag}  test={best_test_tag}  wacc={best_val:.4f}')

preds, labels = all_results[(best_model_tag, best_test_tag)]

DUSHA_LABELS = ['neutral', 'angry', 'positive', 'sad', 'other']

print(f'\n=== Metrics: model={best_model_tag}  test={best_test_tag} ===')
print(f'  accuracy          {accuracy_score(labels, preds):.4f}')
print(f'  weighted_accuracy {balanced_accuracy_score(labels, preds):.4f}')
print(f'  f1_macro          {f1_score(labels, preds, average="macro", zero_division=0):.4f}')
print(f'  f1_weighted       {f1_score(labels, preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(labels, preds, target_names=DUSHA_LABELS, zero_division=0))

In [ ]:
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DUSHA_LABELS, yticklabels=DUSHA_LABELS, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (counts)\nmodel={best_model_tag}  test={best_test_tag}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=DUSHA_LABELS, yticklabels=DUSHA_LABELS, ax=axes[1])
axes[1].set_title('Confusion Matrix (row-normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('/kaggle/working/best_confusion_matrix.png', dpi=150)
plt.show()